# Ranga — Synthetic Dataset Generator

**Model:** [`unsloth/Qwen3.6-27B-MTP-GGUF`](https://huggingface.co/unsloth/Qwen3.6-27B-MTP-GGUF)

| Stage | Who | What |
|---|---|---|
| 1 | Python | MT Samples → service/condition → insurance → tools → job JSON |
| 2 | Qwen | job → `user_query` / `history` / `final_answer` → CSV |

**500** SFT · **200** DPO · **20** eval · Mutuelle / RSSB / Britam


## 0 — Install


In [ ]:
import subprocess, sys

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip_install("datasets", "pandas", "tqdm", "huggingface_hub")
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.check_call(
        "CMAKE_ARGS='-DGGML_CUDA=on' FORCE_CMAKE=1 pip install llama-cpp-python --no-cache-dir -q", shell=True)
elif sys.platform == "darwin":
    subprocess.check_call("CMAKE_ARGS='-DGGML_METAL=on' pip install llama-cpp-python -q", shell=True)
else:
    pip_install("llama-cpp-python")


## 1 — Config & constants


In [ ]:
import copy, csv, json, math, pathlib, random, re, textwrap
from typing import Callable
import pandas as pd

DATASET_DIR = pathlib.Path(".")
GEOJSON_PATH = DATASET_DIR / "health_facilities.geojson"
BRITAM_MD = DATASET_DIR / "InPatient-Medical-Insurance-Contract.md"
OUT_DIR = DATASET_DIR / "ranga_output"
SFT_PATH = OUT_DIR / "ranga_sft_500.csv"
DPO_PATH = OUT_DIR / "ranga_dpo_200.csv"
EVAL_PATH = OUT_DIR / "ranga_eval_20.csv"
PROMPTS_PATH = OUT_DIR / "ranga_generator_jobs.jsonl"
TOOLS_PATH = OUT_DIR / "ranga_tools.json"
CSV_DELIM = "\t"

ALU_LAT, ALU_LON = -1.9695, 30.1589
MAX_RADIUS_KM, TOP_PROVIDERS = 25, 15
LLM_MAX_TOKENS = 220  # JSON output only — keep low for speed
N_SFT, N_DPO, N_EVAL = 500, 200, 20
TRAIN_SCHEMES = ["Mutuelle", "RSSB", "Britam"]
SEED = 42
random.seed(SEED)

TRAINSTYLE_SOURCE = "ranga/synthetic-function-calling-v1"
TRAINSTYLE_SYSTEM_PROMPT = textwrap.dedent('''\
    You are a helpful assistant Zero-Gemma made by ZeroAgency company from Russia. You must be helpful, harmless, and honest.
    Tools (functions) are available. If you decide to invoke one or more of the tools, you must respond with a python list of the function calls.
    Example Format: [func_name1(params_name1=params_value1, params_name2=params_value2...), func_name2(params)]
    Do not use variables. DO NOT USE MARKDOWN SYNTAX. You SHOULD NOT include any other text in the response if you call a function.
    Here is a list of functions in JSON format that you can invoke.
    ''').strip()

SYSTEM_PROMPT = textwrap.dedent('''\
    You are Ranga, an offline hospital recommendation assistant for Rwanda.
    You MUST call tools: getCurrentLocation → getInsuranceCoverageBlock → getNearbyHospitals OR searchHospitalsByCondition → rankHospitalsByPriorityAndCost
    Never guess provider names, network status, or copay amounts. NEVER diagnose. NEVER name drugs.
    ''').strip()

SAFETY_CHECK = re.compile(
    r"\b(diagnos\w+|prescri\w+|medication|drug|tablet|surgery|you have|you may have)\b", re.I)
SPECIALTY_MAP = {
    "Allergy / Immunology": "General Consultation", "Cardiovascular / Pulmonary": "General Consultation",
    "Dentistry": "Dental Care", "Dermatology": "General Consultation",
    "Emergency Room Reports": "Emergency Services", "ENT - Otolaryngology": "General Consultation",
    "Gastroenterology": "Laboratory Tests", "General Medicine": "General Consultation",
    "Hematology - Oncology": "Laboratory Tests", "Neurology": "General Consultation",
    "Obstetrics / Gynecology": "General Consultation", "Ophthalmology": "Optical Care",
    "Orthopedic": "Physical Therapy", "Pain Management": "Physical Therapy",
    "Pathology": "Laboratory Tests", "Pediatrics - Neonatal": "General Consultation",
    "Physical Medicine - Rehab": "Physical Therapy", "Podiatry": "General Consultation",
    "Psychiatry / Psychology": "General Consultation", "Radiology": "Laboratory Tests",
    "Rheumatology": "General Consultation", "Sleep Medicine": "General Consultation",
    "Speech - Language": "Physical Therapy", "Urology": "General Consultation",
}
DISCARD = {"Autopsy", "Neurosurgery", "Surgery", "Bariatrics"}
CRISIS_KEYWORDS = re.compile(r"\b(suicid\w+|self.harm|kill myself|want to die)\b", re.I)
CONDITION_PATH_SERVICES = {"Dental Care", "Optical Care", "Emergency Services", "Physical Therapy"}
OUTPUT_SCHEMA = {
    "user_query": "Short spoken sentence — Rwandan student near ALU asking for hospital routing.",
    "history": "Optional list of 0-1 prior-context strings.",
    "final_answer": "2-4 sentence reply using ONLY ground_truth and tool_results.",
}
DPO_DISTRIBUTION = {"skipped_tool_calls": 60, "out_of_network": 50, "wrong_tier": 40, "hallucinated_cost": 50}

RANGA_TOOLS: list[dict] = []
hospitals: pd.DataFrame
rules_full: pd.DataFrame
rules_in_network: pd.DataFrame
_britam_cache = None


## 2 — MT Samples


In [ ]:
def load_mtsamples() -> pd.DataFrame:
    from datasets import load_dataset
    return load_dataset("harishnair04/mtsamples", split="train").to_pandas()

def extract_context(row) -> dict | None:
    specialty = str(row.get("medical_specialty", "")).strip()
    if specialty in DISCARD or specialty not in SPECIALTY_MAP:
        return None
    desc = str(row.get("description", "")).strip()
    if len(desc) < 20 or CRISIS_KEYWORDS.search(desc):
        return None
    transcription = str(row.get("transcription", "")).strip()
    age_match = re.search(r"(\d+)[- ]year[- ]old", transcription, re.I)
    age_band = "adult"
    if age_match:
        age = int(age_match.group(1))
        age_band = "child" if age < 18 else ("elderly" if age >= 65 else "adult")
    return {
        "description": desc, "medical_specialty": specialty,
        "sample_name": str(row.get("sample_name", "")).strip(),
        "transcription": transcription[:500], "keywords": str(row.get("keywords", "")).strip(),
        "service_category": SPECIALTY_MAP[specialty],
        "condition": str(row.get("sample_name") or desc).strip()[:120],
        "age_band": age_band,
        "has_chronic": bool(re.search(r"history of|chronic|recurrent", desc + " " + transcription, re.I)),
        "source_row_id": int(row.name),
    }

def load_contexts(df: pd.DataFrame | None = None) -> list[dict]:
    df = df if df is not None else load_mtsamples()
    return [c for _, row in df.iterrows() if (c := extract_context(row)) is not None]


## 3 — Geo + hospitals + insurance rules


In [ ]:
def haversine_km(lat1, lon1, lat2, lon2) -> float:
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

def feature_centroid(geometry: dict):
    if geometry["type"] == "Point":
        lon, lat = geometry["coordinates"]
        return lon, lat
    if geometry["type"] == "Polygon":
        ring = geometry["coordinates"][0]
        return sum(c[0] for c in ring) / len(ring), sum(c[1] for c in ring) / len(ring)
    return None, None

def init_data_layer() -> None:
    global hospitals, rules_full, rules_in_network, RANGA_TOOLS
    SERVICE_AMENITY = {
        "pharmacy": ["Pharmacy & Medication"], "clinic": ["General Consultation", "Laboratory Tests"],
        "doctors": ["General Consultation"],
        "hospital": ["General Consultation", "Dental Care", "Optical Care", "Laboratory Tests",
                     "Emergency Services", "Physical Therapy"],
    }
    BASE_FEES = {
        "General Consultation": 15_000, "Dental Care": 25_000, "Optical Care": 30_000,
        "Laboratory Tests": 20_000, "Pharmacy & Medication": 10_000,
        "Emergency Services": 50_000, "Physical Therapy": 20_000,
    }
    SERVICES = list(BASE_FEES.keys())
    SCHEME_NET = {"Mutuelle": "mutuelle", "RSSB": "rssb", "Britam": "britam"}
    SCHEMES = {"Mutuelle": {"rate": 0.10, "ref": True}, "RSSB": {"rate": 0.15}, "Britam": {"rate": 0.10}}
    TIER3 = re.compile(r"king faisal|military|district hospital|teaching", re.I)
    with open(GEOJSON_PATH) as f:
        geo = json.load(f)
    rows = []
    for feat in geo["features"]:
        p = feat["properties"]
        name = (p.get("name") or p.get("name_en") or "").strip()
        amenity = p.get("amenity") or p.get("healthcare") or ""
        if not name or amenity not in SERVICE_AMENITY or p.get("adm1_name") != "Kigali City":
            continue
        lon, lat = feature_centroid(feat["geometry"])
        if lon is None:
            continue
        dist = haversine_km(ALU_LAT, ALU_LON, lat, lon)
        if dist > MAX_RADIUS_KM:
            continue
        tier = "Tier 3 — Referral Hospital" if TIER3.search(name) else "Tier 2 — District Hospital"
        rows.append({"provider_id": p["id"], "name": name, "amenity": amenity, "tier_level": tier,
            "latitude": lat, "longitude": lon, "physical_address": f"{p.get('adm3_name','')}, Kigali",
            "type": "private" if "clinic" in name.lower() or "legacy" in name.lower() else "public",
            "distance_from_alu_km": round(dist, 2)})
    hospitals = pd.DataFrame(rows).sort_values("distance_from_alu_km").drop_duplicates("name").head(TOP_PROVIDERS).reset_index(drop=True)
    records, rid = [], 1
    oon_names = {"Legacy Hospital", "Legacy Clinics"}
    for _, prov in hospitals.iterrows():
        oon = {"Britam"} if prov["name"] in oon_names else set()
        for scheme, meta in SCHEMES.items():
            net = SCHEME_NET[scheme]
            for svc in SERVICES:
                if svc not in SERVICE_AMENITY.get(prov["amenity"], SERVICES) and prov["amenity"] != "hospital":
                    continue
                records.append({"rule_id": rid, "provider_id": prov["provider_id"], "scheme_name": scheme,
                    "network_key": net, "service_category": svc, "out_of_network": int(scheme in oon),
                    "base_fee_rwf": BASE_FEES[svc], "active": 1})
                rid += 1
    rules_full = pd.DataFrame(records).merge(hospitals, on="provider_id")
    acc = {pid: sorted(grp.loc[grp["out_of_network"] == 0, "network_key"].unique()) for pid, grp in rules_full.groupby("provider_id")}
    hospitals["accepted_insurance"] = hospitals["provider_id"].map(acc)
    hospitals["specialties"] = hospitals["amenity"].map(lambda a: ["general medicine"] if a != "hospital" else ["general medicine", "emergency"])
    rules_in_network = rules_full[(rules_full["active"] == 1) & (rules_full["out_of_network"] == 0)].reset_index(drop=True)
    RANGA_TOOLS = [
        {"type": "function", "function": {"name": "getCurrentLocation", "description": "Get student GPS.", "parameters": {"type": "object", "properties": {}}}},
        {"type": "function", "function": {"name": "getInsuranceCoverageBlock", "description": "Insurance block.", "parameters": {"type": "object", "properties": {"insurance": {"type": "string"}}, "required": ["insurance"]}}},
        {"type": "function", "function": {"name": "getNearbyHospitals", "description": "Nearby hospitals.", "parameters": {"type": "object", "properties": {"lat": {"type": "number"}, "lng": {"type": "number"}, "radiusKm": {"type": "number"}, "coverageBlock": {"type": "object"}}, "required": ["lat", "lng", "coverageBlock"]}}},
        {"type": "function", "function": {"name": "searchHospitalsByCondition", "description": "Hospitals by condition.", "parameters": {"type": "object", "properties": {"condition": {"type": "string"}, "coverageBlock": {"type": "object"}, "lat": {"type": "number"}, "lng": {"type": "number"}}, "required": ["condition", "coverageBlock"]}}},
        {"type": "function", "function": {"name": "rankHospitalsByPriorityAndCost", "description": "Rank hospitals.", "parameters": {"type": "object", "properties": {"hospitals": {"type": "array"}, "coverageBlock": {"type": "object"}}, "required": ["hospitals", "coverageBlock"]}}},
    ]
    OUT_DIR.mkdir(exist_ok=True)
    TOOLS_PATH.write_text(json.dumps(RANGA_TOOLS, indent=2))


## 4 — Insurance policy & tool executors


In [ ]:
INSURANCE_BLOCKS = {
    "mutuelle": {"providerName": "Mutuelle de Santé (CBHI)", "networkKey": "mutuelle", "copayPercent": 10.0,
        "requiresReferral": True, "referralNotes": "Referral from local health center required."},
    "rssb": {"providerName": "RSSB / RAMA", "networkKey": "rssb", "copayPercent": 15.0,
        "requiresReferral": False, "referralNotes": "Direct access to certified facilities."},
    "britam": {"providerName": "Britam Private Insurance", "networkKey": "britam", "copayPercent": 10.0,
        "requiresReferral": False, "referralNotes": "Direct billing at partner clinics."},
}
POLICY_EXCERPTS = {
    "Mutuelle": "Mutuelle (CBHI): 10% copay; referral required for Tier 3.",
    "RSSB": "RSSB/RAMA: 15% copay; direct access.",
    "Britam": "",
}

def policy_excerpt(scheme: str) -> str:
    global _britam_cache
    if scheme == "Britam":
        if _britam_cache is None and BRITAM_MD.exists():
            _britam_cache = re.sub(r"\s+", " ", BRITAM_MD.read_text())[:1200]
        return _britam_cache or "Britam: 10% copay at network providers."
    return POLICY_EXCERPTS[scheme]

def tool_getInsuranceCoverageBlock(insurance: str) -> dict:
    c = insurance.lower().replace(" ", "")
    if "mutuelle" in c or "cbhi" in c: return dict(INSURANCE_BLOCKS["mutuelle"])
    if "rssb" in c or "rama" in c: return dict(INSURANCE_BLOCKS["rssb"])
    if "britam" in c: return dict(INSURANCE_BLOCKS["britam"])
    return {"providerName": insurance, "networkKey": "none", "copayPercent": 100.0, "requiresReferral": False}

def hospital_result(prov: pd.Series, dist: float, net_key: str) -> dict:
    accepted = prov.get("accepted_insurance") or []
    net = rules_full[(rules_full["provider_id"] == prov["provider_id"]) & (rules_full["network_key"] == net_key)]
    cost = int(net["base_fee_rwf"].mean()) if not net.empty else 15_000
    return {"hospital": {"id": prov["provider_id"], "name": prov["name"], "address": prov["physical_address"],
        "lat": prov["latitude"], "lng": prov["longitude"], "acceptedInsurance": accepted,
        "emergencyUnit": prov["amenity"] == "hospital", "averageCostRwf": cost,
        "averageCostByInsurance": {net_key: cost}}, "distanceKm": round(dist, 2), "isInNetwork": net_key in accepted}


## 5 — Tool pipeline


In [ ]:
def run_tool_pipeline(scheme: str, service: str, condition: str, use_condition_path: bool) -> dict:
    loc = {"lat": ALU_LAT, "lng": ALU_LON}
    cov = tool_getInsuranceCoverageBlock(scheme)
    net_key = cov["networkKey"]
    if use_condition_path:
        search_tool = "searchHospitalsByCondition"
        results = [hospital_result(prov, haversine_km(loc["lat"], loc["lng"], prov["latitude"], prov["longitude"]), net_key)
                   for _, prov in hospitals.iterrows()]
        pipeline = "condition"
    else:
        search_tool = "getNearbyHospitals"
        results = [hospital_result(prov, d, net_key) for _, prov in hospitals.iterrows()
                   if (d := haversine_km(loc["lat"], loc["lng"], prov["latitude"], prov["longitude"])) <= MAX_RADIUS_KM]
        pipeline = "nearby"
    results.sort(key=lambda x: (not x["isInNetwork"], x["distanceKm"]))
    hosp_res = {"results": results}
    ranked = []
    for res in results:
        h = res["hospital"]
        eff = cov["copayPercent"] if res["isInNetwork"] else 100.0
        base = h["averageCostByInsurance"].get(net_key, h["averageCostRwf"])
        ranked.append({"result": res, "score": 80.0, "estimatedCopayRwf": int(base * eff / 100)})
    ranked.sort(key=lambda x: x["score"], reverse=True)
    rank_res = {"rankedResults": ranked}
    return {
        "pipeline": pipeline,
        "tool_calls": [
            {"name": "getCurrentLocation", "arguments": {}},
            {"name": "getInsuranceCoverageBlock", "arguments": {"insurance": scheme}},
            {"name": search_tool, "arguments": {"condition": condition or service} if use_condition_path else {"lat": loc["lat"], "lng": loc["lng"], "radiusKm": MAX_RADIUS_KM}},
            {"name": "rankHospitalsByPriorityAndCost", "arguments": {"hospital_count": len(results)}},
        ],
        "tool_results": {"getCurrentLocation": loc, "getInsuranceCoverageBlock": cov, search_tool: hosp_res,
                         "rankHospitalsByPriorityAndCost": rank_res},
        "top_pick": ranked[0], "coverage": cov,
    }

def pick_condition_path(service: str) -> bool:
    return service in CONDITION_PATH_SERVICES

def pick_rule(service: str, scheme: str) -> bool:
    m = (rules_in_network["service_category"] == service) & (rules_in_network["scheme_name"] == scheme)
    return not rules_in_network[m].empty


## 6 — Job builder


In [ ]:
def build_job(ctx: dict, scheme: str, job_type: str = "sft") -> dict | None:
    service = ctx["service_category"]
    if not pick_rule(service, scheme):
        return None
    traj = run_tool_pipeline(scheme, service, ctx["condition"], pick_condition_path(service))
    top = traj["top_pick"]
    return {
        "job_type": job_type,
        "mtsample": {"source_row_id": ctx["source_row_id"], "description": ctx["description"],
            "medical_specialty": ctx["medical_specialty"], "sample_name": ctx["sample_name"],
            "transcription_excerpt": ctx["transcription"][:350], "keywords": ctx["keywords"]},
        "scheme": scheme, "service": service, "condition": ctx["condition"],
        "age_band": ctx["age_band"], "has_chronic": ctx["has_chronic"],
        "insurance_policy_excerpt": policy_excerpt(scheme), "tools": RANGA_TOOLS,
        "expected_pipeline": traj["pipeline"], "expected_tool_calls": traj["tool_calls"],
        "tool_results": traj["tool_results"],
        "ground_truth": {"top_hospital": top["result"]["hospital"]["name"],
            "estimated_copay_rwf": top["estimatedCopayRwf"], "in_network": top["result"]["isInNetwork"],
            "distance_km": top["result"]["distanceKm"]},
    }

def build_all_jobs(contexts: list[dict]) -> list[dict]:
    jobs = []
    for ctx in contexts:
        for scheme in TRAIN_SCHEMES:
            if (j := build_job(ctx, scheme)):
                jobs.append(j)
    return jobs

def sample_jobs(jobs: list[dict], n: int, rng: random.Random) -> list[dict]:
    pool = jobs.copy()
    rng.shuffle(pool)
    return pool[:n] if len(pool) >= n else [pool[i % len(pool)] for i in range(n)]


## 7 — Qwen prompt & parse


In [ ]:
def compact_job_for_prompt(job: dict) -> dict:
    """Small prompt payload — never send full hospital lists to Qwen."""
    cov = job["tool_results"]["getInsuranceCoverageBlock"]
    return {
        "mtsample": job["mtsample"],
        "scheme": job["scheme"],
        "service": job["service"],
        "condition": job["condition"],
        "age_band": job["age_band"],
        "insurance_policy_excerpt": job["insurance_policy_excerpt"][:400],
        "coverage": {"provider": cov["providerName"], "copay_pct": cov["copayPercent"]},
        "ground_truth": job["ground_truth"],
        "pipeline": job["expected_pipeline"],
    }

def build_qwen_prompt(job: dict) -> str:
    payload = compact_job_for_prompt(job)
    return (
        "Generate training JSON for a Rwanda hospital-routing assistant.\n"
        "Use ground_truth facts only — do not invent hospitals or prices.\n\n"
        f"INPUT:\n{json.dumps(payload, ensure_ascii=False)}\n\n"
        "Rules: ground user_query in mtsample.description; no diagnosis; no drug names;\n"
        "no hospital names in user_query; final_answer must state copay RWF and mention insurance card.\n"
        f"Return ONLY JSON keys user_query, history, final_answer:\n"
        f"{json.dumps(OUTPUT_SCHEMA, ensure_ascii=False)}"
    )

def parse_qwen_json(raw: str) -> dict | None:
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\s*", "", raw)
        raw = re.sub(r"\s*```$", "", raw)
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if not m: return None
        try: data = json.loads(m.group())
        except json.JSONDecodeError: return None
    if len(data.get("user_query", "")) < 15 or len(data.get("final_answer", "")) < 30:
        return None
    hist = data.get("history") or []
    data["history"] = [hist] if isinstance(hist, str) and hist.strip() else (hist[:1] if isinstance(hist, list) else [])
    return data


## 8 — Assemble messages


In [ ]:
def make_tool_call(name: str, arguments: dict) -> dict:
    return {"type": "function", "function": {"name": name, "arguments": arguments}}

def make_tool_msg(call_id: str, name: str, result: dict) -> dict:
    return {"role": "tool", "tool_call_id": call_id, "name": name, "content": json.dumps(result, ensure_ascii=False)}

def assemble_messages(job: dict, nl: dict) -> list[dict]:
    traj, scheme = job["tool_results"], job["scheme"]
    cov, loc = traj["getInsuranceCoverageBlock"], traj["getCurrentLocation"]
    search_name = "searchHospitalsByCondition" if job["expected_pipeline"] == "condition" else "getNearbyHospitals"
    search_args = ({"condition": job["condition"] or job["service"], "coverageBlock": cov, "lat": loc["lat"], "lng": loc["lng"]}
                   if search_name == "searchHospitalsByCondition"
                   else {"lat": loc["lat"], "lng": loc["lng"], "radiusKm": MAX_RADIUS_KM, "coverageBlock": cov})
    rank_args = {"hospitals": traj[search_name]["results"], "coverageBlock": cov}
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for h in nl.get("history") or []:
        messages += [{"role": "user", "content": f"History: {h}"}, {"role": "assistant", "content": "Noted."}]
    messages.append({"role": "user", "content": nl["user_query"]})
    for step, (tn, args, res) in enumerate([
        ("getCurrentLocation", {}, loc), ("getInsuranceCoverageBlock", {"insurance": scheme}, cov),
        (search_name, search_args, traj[search_name]), ("rankHospitalsByPriorityAndCost", rank_args, traj["rankHospitalsByPriorityAndCost"]),
    ], 1):
        cid = f"call_{step}"
        messages.append({"role": "assistant", "content": None, "tool_calls": [{"id": cid, **make_tool_call(tn, args)}]})
        messages.append(make_tool_msg(cid, tn, res))
    messages.append({"role": "assistant", "content": nl["final_answer"]})
    return messages


## 9 — Records & DPO


In [ ]:
def fallback_nl_from_job(job: dict) -> dict:
    """Deterministic NL if Qwen JSON parse fails — keeps pipeline moving."""
    m, gt, scheme = job["mtsample"], job["ground_truth"], job["scheme"]
    seed = m.get("sample_name") or m.get("description", "")[:80]
    return {
        "user_query": (
            f"I'm a student near ALU with {scheme} insurance and need {job['service'].lower()} "
            f"for something like {seed}."
        ),
        "history": [],
        "final_answer": (
            f"{gt['top_hospital']} ({gt['distance_km']} km) works for {job['service']} under {scheme}. "
            f"Estimated copay: {gt['estimated_copay_rwf']:,} RWF. Bring your insurance card."
        ),
    }

def job_to_record(job: dict, nl: dict) -> dict:
    return {"tools": job["tools"], "messages": assemble_messages(job, nl),
        "metadata": {"source_row_id": job["mtsample"]["source_row_id"], "scheme": job["scheme"],
            "service": job["service"], "pipeline": job["expected_pipeline"],
            "top_provider": job["ground_truth"]["top_hospital"], "oop_rwf": job["ground_truth"]["estimated_copay_rwf"]}}

def generate_record_from_job(job: dict, llm_fn: Callable[..., str], *, allow_fallback: bool = True) -> dict | None:
    raw = llm_fn(build_qwen_prompt(job), max_tokens=LLM_MAX_TOKENS, temp=0.5)
    nl = parse_qwen_json(raw) or (fallback_nl_from_job(job) if allow_fallback else None)
    if nl is None or SAFETY_CHECK.search(nl["user_query"] + nl["final_answer"]):
        return None
    return job_to_record(job, nl)

def generate_eval_record_from_job(job: dict, llm_fn: Callable[..., str], *, allow_fallback: bool = True) -> dict | None:
    raw = llm_fn(build_qwen_prompt(job), max_tokens=LLM_MAX_TOKENS, temp=0.5)
    nl = parse_qwen_json(raw) or (fallback_nl_from_job(job) if allow_fallback else None)
    if nl is None or SAFETY_CHECK.search(nl["user_query"] + nl.get("final_answer", "")):
        return None
    g = job["ground_truth"]
    return {"query": nl["user_query"], "tools": job["tools"],
        "expected_pipeline": job["expected_pipeline"], "expected_tool_calls": job["expected_tool_calls"],
        "expected_service": job["service"], "expected_scheme": job["scheme"], "expected_severity": "low",
        "history_len": len(nl.get("history") or []), "expected_provider": g["top_hospital"],
        "expected_oop_rwf": g["estimated_copay_rwf"], "source_row_id": job["mtsample"]["source_row_id"],
        "ground_truth_label": "correct_tool_trajectory"}

def corrupt_record(chosen: dict, flaw: str) -> dict | None:
    rejected = copy.deepcopy(chosen)
    msgs, final = rejected["messages"], rejected["messages"][-1]["content"]
    oop_m = re.search(r"(\d[\d,]+) RWF", final or "")
    oop = int(oop_m.group(1).replace(",", "")) if oop_m else 15_000
    if flaw == "skipped_tool_calls":
        rejected["messages"] = [msgs[0], msgs[-2], {"role": "assistant", "content": final}]
    elif flaw == "out_of_network":
        rejected["messages"][-1]["content"] = f"Go to Legacy Hospital. {final}"
    elif flaw == "wrong_tier":
        rejected["messages"][-1]["content"] = "No referral needed. " + final
    elif flaw == "hallucinated_cost":
        rejected["messages"][-1]["content"] = re.sub(r"\d[\d,]+ RWF", f"{oop // 2:,} RWF", final, count=1)
    else:
        return None
    return rejected


## 10 — CSV export


In [ ]:
def fmt_args(arguments: dict) -> str:
    return ", ".join(f"{k}={json.dumps(v, ensure_ascii=False)}" for k, v in (arguments or {}).items())

def render_trainstyle_conversation(messages: list[dict], tools: list[dict]) -> str:
    lines = ["<bos><start_of_turn>system", TRAINSTYLE_SYSTEM_PROMPT, json.dumps(tools, ensure_ascii=False, indent=2), "<end_of_turn>"]
    for m in messages:
        role = m.get("role")
        if role == "system": continue
        if role == "user":
            lines += ["<start_of_turn>user", (m.get("content") or "").rstrip(), "<end_of_turn>"]
        elif role == "assistant" and m.get("tool_calls"):
            calls = [f"{tc['function']['name']}({fmt_args(tc['function'].get('arguments', {}))})" for tc in m["tool_calls"]]
            lines += ["<start_of_turn>model", "[" + ", ".join(calls) + "]", "<end_of_turn>"]
        elif role == "tool":
            lines += ["<start_of_turn>user", "<tool_response>", m.get("content") or "{}", "</tool_response><end_of_turn>"]
        elif role == "assistant":
            lines += ["<start_of_turn>model", (m.get("content") or "").rstrip(), "<end_of_turn>"]
    return "\n".join(lines)

def save_sft_csv(path: pathlib.Path, records: list[dict]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, delimiter=CSV_DELIM)
        for rec in records:
            tools = rec.get("tools", RANGA_TOOLS)
            w.writerow([json.dumps(rec["messages"], ensure_ascii=False), json.dumps(tools, ensure_ascii=False),
                        render_trainstyle_conversation(rec["messages"], tools), TRAINSTYLE_SOURCE])


In [ ]:
def save_dpo_csv(path: pathlib.Path, pairs: list[dict]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, delimiter=CSV_DELIM)
        for pair in pairs:
            tools = pair.get("tools", RANGA_TOOLS)
            w.writerow([json.dumps(pair["chosen"]["messages"], ensure_ascii=False),
                        json.dumps(pair["rejected"]["messages"], ensure_ascii=False),
                        json.dumps(tools, ensure_ascii=False), pair["rejection_type"],
                        json.dumps(pair.get("metadata", {}), ensure_ascii=False), TRAINSTYLE_SOURCE])

def save_eval_csv(path: pathlib.Path, records: list[dict]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, delimiter=CSV_DELIM)
        for rec in records:
            w.writerow([rec["query"], json.dumps(rec.get("tools", RANGA_TOOLS), ensure_ascii=False),
                rec["expected_pipeline"], json.dumps(rec["expected_tool_calls"], ensure_ascii=False),
                rec["expected_service"], rec["expected_scheme"], rec.get("expected_severity", "low"),
                rec.get("history_len", 0), rec["expected_provider"], rec["expected_oop_rwf"],
                rec["source_row_id"], rec.get("ground_truth_label", "correct_tool_trajectory")])

def save_jobs_jsonl(path: pathlib.Path, jobs: list[dict]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for j in jobs:
            f.write(json.dumps(j, ensure_ascii=False) + "\n")


## 11 — Build jobs


In [ ]:
init_data_layer()
contexts = load_contexts()
all_jobs = build_all_jobs(contexts)
rng = random.Random(SEED)
rng.shuffle(all_jobs)
save_jobs_jsonl(PROMPTS_PATH, all_jobs)
sft_jobs = sample_jobs(all_jobs, N_SFT, rng)
eval_jobs = sample_jobs(all_jobs, N_EVAL, random.Random(SEED + 1))
print(f"Contexts {len(contexts)} | jobs {len(all_jobs)} | SFT pool {len(sft_jobs)} | eval pool {len(eval_jobs)}")


## 12 — Load Qwen


In [ ]:
import os
from huggingface_hub import login
from llama_cpp import Llama

MODEL_REPO = "unsloth/Qwen3.6-27B-MTP-GGUF"
GGUF_FILE = "Qwen3.6-27B-UD-Q4_K_XL.gguf"
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)

print(f"Loading {MODEL_REPO} …")
llm_model = Llama.from_pretrained(repo_id=MODEL_REPO, filename=GGUF_FILE, token=hf_token,
    n_ctx=4096, n_gpu_layers=-1, n_batch=512, verbose=False)
print("Qwen ready.")

def strip_thinking(text: str) -> str:
    close = "\u003c/" + "think" + "\u003e"
    return text.split(close, 1)[-1].strip() if close in text else text.strip()

def llm(prompt: str, max_tokens: int = LLM_MAX_TOKENS, temp: float = 0.5) -> str:
    out = llm_model.create_chat_completion(
        messages=[{"role": "user", "content": prompt}], max_tokens=max_tokens, temperature=temp, top_p=0.9)
    return strip_thinking(out["choices"][0]["message"].get("content") or "")


## 13 — Generate SFT


In [ ]:
import time
from tqdm.auto import tqdm

t0 = time.time()
smoke = generate_record_from_job(sft_jobs[0], llm)
print(f"Smoke test: {'OK' if smoke else 'FAIL'} in {time.time() - t0:.1f}s")
if smoke is None:
    raise RuntimeError("Smoke test failed — check Qwen load / job pool before running 500.")

sft_records, idx, fails = [], 0, 0
pbar = tqdm(total=N_SFT, desc="SFT")
max_attempts = max(N_SFT * 2, len(sft_jobs) * 4)
while len(sft_records) < N_SFT and idx < max_attempts:
    job = sft_jobs[idx % len(sft_jobs)]
    idx += 1
    t0 = time.time()
    rec = generate_record_from_job(job, llm)
    elapsed = time.time() - t0
    if rec is None:
        fails += 1
        if fails <= 3:
            pbar.write(f"attempt {idx} rejected ({elapsed:.1f}s)")
        continue
    sft_records.append(rec)
    pbar.update(1)
    if len(sft_records) == 1:
        pbar.write(f"First record in {elapsed:.1f}s")
pbar.close()
if len(sft_records) < N_SFT:
    print(f"WARNING: only {len(sft_records)}/{N_SFT} after {idx} attempts ({fails} rejects)")
random.shuffle(sft_records)
save_sft_csv(SFT_PATH, sft_records)
print(f"Saved {len(sft_records)} → {SFT_PATH.name}")


## 14 — DPO + Eval


In [ ]:
from tqdm.auto import tqdm
dpo_pairs, pool_i = [], 0
for flaw, target in DPO_DISTRIBUTION.items():
    got = 0
    pbar = tqdm(total=target, desc=flaw)
    while got < target and pool_i < len(sft_records) * 3:
        chosen = sft_records[pool_i % len(sft_records)]
        pool_i += 1
        rejected = corrupt_record(chosen, flaw)
        if rejected is None: continue
        dpo_pairs.append({"tools": RANGA_TOOLS, "chosen": chosen, "rejected": rejected,
            "rejection_type": flaw, "metadata": chosen["metadata"]})
        got += 1
        pbar.update(1)
    pbar.close()
save_dpo_csv(DPO_PATH, dpo_pairs)
print(f"DPO saved {len(dpo_pairs)} → {DPO_PATH.name}")

import time
t0 = time.time()
smoke_eval = generate_eval_record_from_job(eval_jobs[0], llm)
print(f"Eval smoke test: {'OK' if smoke_eval else 'FAIL'} in {time.time() - t0:.1f}s")
if smoke_eval is None:
    raise RuntimeError("Eval smoke test failed — check Qwen / eval_jobs before running.")

eval_records, idx, fails = [], 0, 0
pbar = tqdm(total=N_EVAL, desc="Eval")
max_eval_attempts = max(N_EVAL * 2, len(eval_jobs) * 4)
while len(eval_records) < N_EVAL and idx < max_eval_attempts:
    job = eval_jobs[idx % len(eval_jobs)]
    idx += 1
    t0 = time.time()
    rec = generate_eval_record_from_job(job, llm)
    elapsed = time.time() - t0
    if rec is None:
        fails += 1
        if fails <= 3:
            pbar.write(f"eval attempt {idx} rejected ({elapsed:.1f}s)")
        continue
    eval_records.append(rec)
    pbar.update(1)
    if len(eval_records) == 1:
        pbar.write(f"First eval in {elapsed:.1f}s")
pbar.close()
if len(eval_records) < N_EVAL:
    print(f"WARNING: only {len(eval_records)}/{N_EVAL} eval after {idx} attempts ({fails} rejects)")
save_eval_csv(EVAL_PATH, eval_records)
print(f"DPO {len(dpo_pairs)} | Eval {len(eval_records)} → {EVAL_PATH.name}")


## 15 — Validation


In [ ]:
def extract_tool_names(messages):
    return [tc["function"]["name"] for m in messages for tc in (m.get("tool_calls") or [])]

def count_with_rank(records):
    return sum(1 for r in records if "rankHospitalsByPriorityAndCost" in extract_tool_names(r["messages"]))

print(f"SFT {len(sft_records)} | rank_ok {count_with_rank(sft_records)}/{len(sft_records)}")
print(f"DPO {len(dpo_pairs)} | Eval {len(eval_records)} | schemes {TRAIN_SCHEMES}")
